In [20]:
import csv

In [21]:
def load_data(filename):
    student_data = []
    subjects = []

    try:
        with open(filename, mode='r', encoding='utf-8-sig') as file:
            reader = csv.reader(file)

            try:
                header = next(reader)
                subjects = header[1:]
            except StopIteration:
                print("Error: The file is empty.")
                return [], []

            for row in reader:
                if not row:
                    continue

                name = row[0]
                scores = {}
                is_valid_row = True

                for i in range(len(subjects)):
                    try:
                        scores[subjects[i]] = int(row[i+1])
                    except (ValueError, IndexError):
                        is_valid_row = False
                        break

                if is_valid_row:
                    student_data.append({"name": name, "scores": scores})
                else:
                    print(f"Skipping invalid data for entry: {name}")

    except FileNotFoundError:
        print(f"Error: Could not find '{filename}'.")

    return subjects, student_data

In [22]:
def get_grade(average):
    if average >= 70: return 'A'
    elif average >= 60: return 'B'
    elif average >= 50: return 'C'
    elif average >= 45: return 'D'
    else: return 'F'

In [23]:
def process_students(student_data):
    processed = []
    pass_count = 0
    fail_count = 0

    for student in student_data:
        scores = list(student['scores'].values())
        average = sum(scores) / len(scores)
        grade = get_grade(average)

        if average >= 50:
            pass_count += 1
        else:
            fail_count += 1

        processed.append({
            "name": student['name'],
            "average": average,
            "grade": grade
        })

    processed.sort(key=lambda x: x['average'], reverse=True)
    return processed, pass_count, fail_count

In [24]:
def process_subjects(student_data, subjects):
    subject_stats = []

    for subject in subjects:
        all_scores = [student['scores'][subject] for student in student_data]
        avg_score = sum(all_scores) / len(all_scores)

        subject_stats.append({
            "subject": subject,
            "average": avg_score,
            "highest": max(all_scores),
            "lowest": min(all_scores)
        })

    subject_stats.sort(key=lambda x: x['average'])
    return subject_stats

In [25]:
def generate_report(students, subject_stats, pass_count, fail_count):
    report = []

    report.append("STUDENT RESULT PROCESSOR — REPORT")
    report.append("======================================================\n")

    report.append("1) PER-STUDENT AVERAGE & GRADE\n")
    for s in students:
        report.append(f"   {s['name']:<20} avg={s['average']:<5.2f}  grade={s['grade']}")
    report.append("\n")

    report.append("2) CLASS AVERAGE PER SUBJECT\n")
    for sub in sorted(subject_stats, key=lambda x: x['subject']):
        report.append(f"   {sub['subject']:<15} class avg = {sub['average']:.2f}")
    report.append("\n")

    top_student = students[0]
    report.append("3) TOP STUDENT (highest overall average)\n")
    report.append(f"   {top_student['name']} ({top_student['average']:.2f})\n")

    report.append("4) PASS / FAIL COUNT (pass = overall avg >= 50)\n")
    report.append(f"   Passed: {pass_count}    Failed: {fail_count}\n")

    report.append("5) STUDENT LEADERBOARD (highest to lowest average)\n")
    for i, s in enumerate(students, 1):
        report.append(f"   {i}. {s['name']:<20} {s['average']:.2f}  ({s['grade']})")
    report.append("\n")

    report.append("6) SUBJECTS RANKED HARDEST TO EASIEST (lowest avg = hardest)\n")
    for i, sub in enumerate(subject_stats, 1):
        note = "  <- hardest" if i == 1 else "  <- easiest" if i == len(subject_stats) else ""
        report.append(f"   {i}. {sub['subject']:<15} {sub['average']:.2f}{note}")

    return "\n".join(report)

In [26]:
def interactive_lookup(subject_stats):
    print("\n" + "="*54)
    print("SUBJECT LOOKUP TOOL")
    print("="*54)

    subject_names = [s['subject'] for s in subject_stats]

    while True:
        query = input(f"Enter a subject to look up ({', '.join(subject_names)}) or 'quit' to exit: ").strip()

        if query.lower() == 'quit':
            print("Exiting lookup. Project complete!")
            break

        matched_stat = next((s for s in subject_stats if s['subject'].lower() == query.lower()), None)

        if matched_stat:
            print(f"\n   --- {matched_stat['subject'].upper()} SUMMARY ---")
            print(f"   Class Average : {matched_stat['average']:.2f}")
            print(f"   Highest Score : {matched_stat['highest']}")
            print(f"   Lowest Score  : {matched_stat['lowest']}\n")
        else:
            print("\nSubject not found. Please check your spelling and try again.\n")

In [27]:
def main():
    filename = "results.csv"

    subjects, raw_data = load_data(filename)
    if not raw_data:
        return

    students, pass_count, fail_count = process_students(raw_data)
    subject_stats = process_subjects(raw_data, subjects)

    report_text = generate_report(students, subject_stats, pass_count, fail_count)
    print(report_text)

    with open("report.txt", "w", encoding='utf-8') as file:
        file.write(report_text)
    print("\n[!] Report successfully saved to 'report.txt'.")

    interactive_lookup(subject_stats)

if __name__ == "__main__":
    main()

STUDENT RESULT PROCESSOR — REPORT

1) PER-STUDENT AVERAGE & GRADE

   Amina Bello          avg=89.60  grade=A
   Ibrahim Musa         avg=81.60  grade=A
   Fatima Sani          avg=73.00  grade=A
   Chidi Okafor         avg=71.00  grade=A
   Ngozi Eze            avg=69.20  grade=B
   Blessing Effiong     avg=57.80  grade=C
   Tunde Adeyemi        avg=46.20  grade=D
   Emeka Nwankwo        avg=41.60  grade=F


2) CLASS AVERAGE PER SUBJECT

   Biology         class avg = 67.50
   Chemistry       class avg = 64.75
   English         class avg = 67.88
   Mathematics     class avg = 66.75
   Physics         class avg = 64.38


3) TOP STUDENT (highest overall average)

   Amina Bello (89.60)

4) PASS / FAIL COUNT (pass = overall avg >= 50)

   Passed: 6    Failed: 2

5) STUDENT LEADERBOARD (highest to lowest average)

   1. Amina Bello          89.60  (A)
   2. Ibrahim Musa         81.60  (A)
   3. Fatima Sani          73.00  (A)
   4. Chidi Okafor         71.00  (A)
   5. Ngozi Eze         